# Fase B — Entrenamiento del modelo CIL (Conditional Imitation Learning)

Behavioral Cloning **condicionado por comandos de navegación**, según Codevilla
et al. (CLIymodelos.pdf). Backbone convolucional estilo **PilotNet / Bojarski**
compartido, y una **rama (branch) por comando**; el comando activo selecciona la
rama cuya salida es el ángulo de dirección.

- Entrada: imagen de la cámara a bordo + comando de navegación (one-hot).
- Salida: `steering` (un escalar). **La velocidad NO se entrena** (es constante).
- Comandos: `2=FOLLOW`, `3=LEFT`, `4=RIGHT` (**3 ramas**). El comando
  `5=STRAIGHT` no tiene rama propia: en una ciudad con líneas de carril, "cruzar
  derecho" se comporta igual que seguir el carril, así que el controlador de
  inferencia **enruta STRAIGHT a FOLLOW**.

> Las etiquetas de comando provienen de `training/relabel_from_gps.py`, que las
> deriva de la geometría real de la trayectoria GPS (las anotadas con el teclado
> durante la conducción quedaron mal). El steering es el humano, sin tocar.

> Recomendado: ejecutar en **Google Colab (GPU)** o en un venv con **Python
> 3.11/3.12 + TensorFlow**. TensorFlow aún no publica wheels para Python 3.14.

In [ ]:
# === Paso 0 (Colab): traer el proyecto (incluye el dataset) desde GitHub. ===
# El repo debe ser PÚBLICO para clonar sin token (si es privado, usá un PAT).
REPO_URL = "https://github.com/cesarcamilov1/navegacion-autonoma-proyecto-final-equipo-21.git"
get_ipython().system('git clone $REPO_URL')

# El dataset está DENTRO del repo, en la subcarpeta 'dataset'. En la celda de
# Configuración dejá:
#   DATASET_DIR = "navegacion-autonoma-proyecto-final-equipo-21/dataset"

## 1. Dependencias

In [ ]:
# En Colab normalmente ya están. En local:
# pip install tensorflow opencv-python pandas scikit-learn matplotlib
import sys, os, json, math, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATASET_DIR = "dataset"                       # carpeta con img/ y el CSV
# Usamos el CSV RE-ETIQUETADO por geometría GPS (training/relabel_from_gps.py):
# corrige los comandos (que se anotaron mal con el teclado) y normaliza las
# rutas de imagen a '/' para que funcionen en Colab/Linux.
CSV_PATH    = os.path.join(DATASET_DIR, "driving_log_relabeled.csv")

# La cámara entrega 320x160. Recortamos el cielo (arriba) y un poco abajo, y
# redimensionamos a la entrada del modelo (estilo Codevilla 200x88).
CROP_TOP    = 55      # px recortados arriba (cielo / horizonte)
CROP_BOTTOM = 5       # px recortados abajo (cofre / borde)
IMG_W       = 200
IMG_H       = 88

# Modelo de 3 comandos: FOLLOW, LEFT, RIGHT. STRAIGHT (5) NO tiene rama propia
# (no hay datos fiables y se comporta como FOLLOW): el controlador de inferencia
# enruta el comando STRAIGHT a FOLLOW.
COMMANDS    = [2, 3, 4]                        # FOLLOW, LEFT, RIGHT
CMD_INDEX   = {c: i for i, c in enumerate(COMMANDS)}
N_CMD       = len(COMMANDS)

BATCH       = 64
EPOCHS      = 60
LR          = 1e-4
VAL_SPLIT   = 0.15
MAX_STEER   = 0.5                              # mismo tope que el recolector
# Balanceo por comando: muestras de cada comando por época. Con el dataset
# combinado (~35k, FOLLOW mayoritario) un tope evita épocas enormes en Colab
# sin perder el balance. None = usar el tamaño de la clase mayor.
BALANCE_CAP = 8000

OUT_MODEL   = "cil_model.keras"
OUT_CONFIG  = "model_config.json"

In [ ]:
DATASET_DIR = "dataset"                       # carpeta con img/ y driving_log.csv
CSV_PATH    = os.path.join(DATASET_DIR, "driving_log.csv")

# La cámara entrega 320x160. Recortamos el cielo (arriba) y un poco abajo, y
# redimensionamos a la entrada del modelo (estilo Codevilla 200x88).
CROP_TOP    = 55      # px recortados arriba (cielo / horizonte)
CROP_BOTTOM = 5       # px recortados abajo (cofre / borde)
IMG_W       = 200
IMG_H       = 88

COMMANDS    = [2, 3, 4, 5]                     # FOLLOW, LEFT, RIGHT, STRAIGHT
CMD_INDEX   = {c: i for i, c in enumerate(COMMANDS)}
N_CMD       = len(COMMANDS)

BATCH       = 64
EPOCHS      = 60
LR          = 1e-4
VAL_SPLIT   = 0.15
MAX_STEER   = 0.5                              # mismo tope que el recolector

OUT_MODEL   = "cil_model.keras"
OUT_CONFIG  = "model_config.json"

## 3. Carga del CSV y análisis del dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
# Une rutas relativas con la carpeta del dataset.
df["path"] = df["image"].apply(lambda p: os.path.join(DATASET_DIR, p))
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
print("Filas válidas:", len(df))
print(df["command_name"].value_counts())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["command_name"].value_counts().plot(kind="bar", ax=ax[0], title="Comandos")
ax[1].hist(df["steering"], bins=50); ax[1].set_title("Distribución de steering")
plt.tight_layout(); plt.show()

## 4. Preprocesamiento

`preprocess()` debe ser idéntico en entrenamiento y en inferencia. Recibe una
imagen **BGR** (como la entrega OpenCV / la cámara de Webots) y devuelve un
tensor RGB normalizado al tamaño del modelo.

In [ ]:
def preprocess(img_bgr):
    """BGR (HxWx3) -> RGB recortada y redimensionada a (IMG_H, IMG_W, 3)."""
    h = img_bgr.shape[0]
    img = img_bgr[CROP_TOP:h - CROP_BOTTOM, :, :]
    img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32)

# Muestra de ejemplo
sample = cv2.imread(df["path"].iloc[0])
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1); plt.title("original"); plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
plt.subplot(1, 2, 2); plt.title("preprocesada"); plt.imshow(preprocess(sample).astype(np.uint8))
plt.tight_layout(); plt.show()

FLIP_SWAP = {2: 2, 3: 4, 4: 3}   # LEFT<->RIGHT al espejar (FOLLOW queda igual)
SHIFT_MAX_PX = 24
SHIFT_STEER_PER_PX = 0.004

def augment(img_bgr, steering, command):
    # Flip
    if random.random() < 0.5:
        img_bgr = cv2.flip(img_bgr, 1)
        steering = -steering
        command = FLIP_SWAP[command]
    # Brillo
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] *= random.uniform(0.5, 1.3)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
    img_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    # Desplazamiento horizontal + corrección de steering
    if random.random() < 0.5:
        dx = random.randint(-SHIFT_MAX_PX, SHIFT_MAX_PX)
        M = np.float32([[1, 0, dx], [0, 1, 0]])
        img_bgr = cv2.warpAffine(img_bgr, M, (img_bgr.shape[1], img_bgr.shape[0]),
                                 borderMode=cv2.BORDER_REPLICATE)
        steering = steering - dx * SHIFT_STEER_PER_PX
    steering = float(np.clip(steering, -MAX_STEER, MAX_STEER))
    return img_bgr, steering, command

In [ ]:
FLIP_SWAP = {2: 2, 3: 4, 4: 3, 5: 5}   # LEFT<->RIGHT al espejar
SHIFT_MAX_PX = 24
SHIFT_STEER_PER_PX = 0.004

def augment(img_bgr, steering, command):
    # Flip
    if random.random() < 0.5:
        img_bgr = cv2.flip(img_bgr, 1)
        steering = -steering
        command = FLIP_SWAP[command]
    # Brillo
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] *= random.uniform(0.5, 1.3)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
    img_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    # Desplazamiento horizontal + corrección de steering
    if random.random() < 0.5:
        dx = random.randint(-SHIFT_MAX_PX, SHIFT_MAX_PX)
        M = np.float32([[1, 0, dx], [0, 1, 0]])
        img_bgr = cv2.warpAffine(img_bgr, M, (img_bgr.shape[1], img_bgr.shape[0]),
                                 borderMode=cv2.BORDER_REPLICATE)
        steering = steering - dx * SHIFT_STEER_PER_PX
    steering = float(np.clip(steering, -MAX_STEER, MAX_STEER))
    return img_bgr, steering, command

class CILSequence(keras.utils.Sequence):
    """Generador con BALANCEO por comando en entrenamiento.

    El dataset está dominado por FOLLOW (~78%); LEFT/RIGHT son minoría. Como
    recomienda Codevilla ("minibatches con igual número de muestras por
    comando"), en cada época tomamos `BALANCE_CAP` muestras de cada comando
    (sobremuestreando la minoría y submuestreando FOLLOW). `on_epoch_end`
    re-sortea, así que a lo largo de las épocas se cubre todo FOLLOW. La
    augmentation aleatoria hace que las repeticiones de la minoría NO sean
    idénticas. En validación NO se balancea ni se aumenta (distribución real).
    """
    def __init__(self, frame, batch=BATCH, training=True, **kw):
        super().__init__(**kw)
        self.df = frame.reset_index(drop=True)
        self.batch = batch
        self.training = training
        cmds = self.df["command"].values
        self.groups = {int(c): np.where(cmds == c)[0] for c in np.unique(cmds)}
        self.on_epoch_end()

    def on_epoch_end(self):
        if self.training:
            sizes = [len(g) for g in self.groups.values()]
            target = max(sizes) if BALANCE_CAP is None else min(max(sizes), BALANCE_CAP)
            parts = [np.random.choice(g, target, replace=len(g) < target)
                     for g in self.groups.values()]
            self.idx = np.concatenate(parts)
            np.random.shuffle(self.idx)
        else:
            self.idx = np.arange(len(self.df))

    def __len__(self):
        return math.ceil(len(self.idx) / self.batch)

    def __getitem__(self, i):
        rows = self.idx[i * self.batch:(i + 1) * self.batch]
        X_img, X_cmd, y = [], [], []
        for r in rows:
            row = self.df.iloc[r]
            img = cv2.imread(row["path"])
            if img is None:
                continue
            steering = float(row["steering"]); command = int(row["command"])
            if self.training:
                img, steering, command = augment(img, steering, command)
            X_img.append(preprocess(img))
            onehot = np.zeros(N_CMD, np.float32); onehot[CMD_INDEX[command]] = 1.0
            X_cmd.append(onehot); y.append(steering)
        return ({"image": np.array(X_img, np.float32),
                 "command": np.array(X_cmd, np.float32)},
                np.array(y, np.float32))

In [ ]:
class CILSequence(keras.utils.Sequence):
    def __init__(self, frame, batch=BATCH, training=True, **kw):
        super().__init__(**kw)
        self.df = frame.reset_index(drop=True)
        self.batch = batch
        self.training = training
        self.idx = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.df) / self.batch)

    def on_epoch_end(self):
        if self.training:
            np.random.shuffle(self.idx)

    def __getitem__(self, i):
        rows = self.idx[i * self.batch:(i + 1) * self.batch]
        X_img, X_cmd, y = [], [], []
        for r in rows:
            row = self.df.iloc[r]
            img = cv2.imread(row["path"])
            if img is None:
                continue
            steering = float(row["steering"]); command = int(row["command"])
            if self.training:
                img, steering, command = augment(img, steering, command)
            X_img.append(preprocess(img))
            onehot = np.zeros(N_CMD, np.float32); onehot[CMD_INDEX[command]] = 1.0
            X_cmd.append(onehot); y.append(steering)
        return ({"image": np.array(X_img, np.float32),
                 "command": np.array(X_cmd, np.float32)},
                np.array(y, np.float32))

## 7. Split train / val

In [ ]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=VAL_SPLIT, random_state=SEED,
                                    stratify=df["command"])
print("train:", len(train_df), "| val:", len(val_df))
train_seq = CILSequence(train_df, training=True)
val_seq   = CILSequence(val_df, training=False)

def build_cil_model():
    img_in = keras.Input(shape=(IMG_H, IMG_W, 3), name="image")
    cmd_in = keras.Input(shape=(N_CMD,), name="command")

    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(img_in)
    x = layers.Conv2D(24, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(36, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(48, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(100, activation="relu")(x)
    x = layers.Dense(50, activation="relu")(x)

    branch_outs = []
    for i in range(N_CMD):
        b = layers.Dense(25, activation="relu", name=f"head_{i}")(x)
        b = layers.Dense(1, name=f"steer_{i}")(b)
        branch_outs.append(b)
    all_branches = layers.Concatenate(name="branches")(branch_outs)   # (None, N_CMD)
    # Producto punto con el one-hot = selecciona la rama del comando activo.
    steer = layers.Dot(axes=1, name="steering")([all_branches, cmd_in])

    model = keras.Model([img_in, cmd_in], steer, name="CIL_branched")
    model.compile(optimizer=keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
    return model

model = build_cil_model()
model.summary()

In [ ]:
def build_cil_model():
    img_in = keras.Input(shape=(IMG_H, IMG_W, 3), name="image")
    cmd_in = keras.Input(shape=(N_CMD,), name="command")

    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(img_in)
    x = layers.Conv2D(24, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(36, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(48, 5, strides=2, activation="relu")(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(100, activation="relu")(x)
    x = layers.Dense(50, activation="relu")(x)

    branch_outs = []
    for i in range(N_CMD):
        b = layers.Dense(25, activation="relu", name=f"head_{i}")(x)
        b = layers.Dense(1, name=f"steer_{i}")(b)
        branch_outs.append(b)
    all_branches = layers.Concatenate(name="branches")(branch_outs)   # (None, 4)
    # Producto punto con el one-hot = selecciona la rama del comando activo.
    steer = layers.Dot(axes=1, name="steering")([all_branches, cmd_in])

    model = keras.Model([img_in, cmd_in], steer, name="CIL_branched")
    model.compile(optimizer=keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
    return model

model = build_cil_model()
model.summary()

## 9. Entrenamiento

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(OUT_MODEL, monitor="val_loss",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=4, min_lr=1e-6, verbose=1),
]
history = model.fit(train_seq, validation_data=val_seq,
                    epochs=EPOCHS, callbacks=callbacks)

## 10. Curvas de aprendizaje

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val"); plt.title("MSE"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history.history["mae"], label="train")
plt.plot(history.history["val_mae"], label="val"); plt.title("MAE"); plt.legend()
plt.tight_layout(); plt.show()

## 11. Predicciones de ejemplo (sanity check)

In [ ]:
xb, yb = val_seq[0]
pred = model.predict(xb, verbose=0).ravel()
for i in range(min(8, len(yb))):
    cmd = COMMANDS[int(np.argmax(xb["command"][i]))]
    print(f"cmd={cmd}  real={yb[i]:+.3f}  pred={pred[i]:+.3f}")

## 12. Exportar modelo + configuración

El `model_config.json` lleva los parámetros de preprocesamiento para que el
controlador de inferencia (Fase C) reproduzca EXACTAMENTE la misma entrada.

In [ ]:
model.save(OUT_MODEL)
config = {
    "crop_top": CROP_TOP, "crop_bottom": CROP_BOTTOM,
    "img_w": IMG_W, "img_h": IMG_H,
    "commands": COMMANDS, "max_steer": MAX_STEER,
}
with open(OUT_CONFIG, "w") as f:
    json.dump(config, f, indent=2)
print("Guardado:", OUT_MODEL, "y", OUT_CONFIG)